# EP.03.01. Streaming 

**Materia:** Manejo Masivo de Datos  
**Tecnología:** Apache Spark 3.5.3 · PySpark Structured Streaming · OpenBSD Netcat (`nc`)  
**Entorno:** Ubuntu Linux · Python 3.12 · JupyterLab  

### Objetivo de la Práctica
Comprender e implementar el procesamiento de flujos de datos en **tiempo real** mediante **Spark Structured Streaming**, conectando un origen continuo de texto por sockets TCP (`localhost:9999`) y analizando las diferencias entre transformaciones con y sin estado, modos de salida (`complete`, `update`, `append`) y control de frecuencia mediante *triggers*.

## 1. Inicialización de la Sesión de Spark Streaming

Para procesar datos en tiempo real de forma local necesitamos configurar `SparkSession` con al menos **2 hilos de ejecución** (`local[2]`): uno queda dedicado exclusivamente a escuchar y recibir los datos entrantes desde el puerto de red, mientras que el segundo procesa cada micro-lote (*micro-batch*). Además, silenciamos los registros informativos de la JVM para mantener limpia la salida de consola.

In [2]:
import os
import logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

# Silenciar logs informativos de JVM/Spark para salidas limpias
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=false pyspark-shell"
logging.getLogger("py4j").setLevel(logging.ERROR)

# Crear SparkSession con 2 hilos (1 receptor + 1 procesador, mínimo para streaming)
spark = (
    SparkSession.builder
    .appName("Streaming IABD WordCount")
    .master("local[2]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Streaming listo | Versión: {spark.version} | Master: {spark.sparkContext.master}")

Spark Streaming listo | Versión: 3.5.3 | Master: local[2]


## 2. Caso 1: Hola Spark Streaming (`WordCount` en tiempo real)

En esta etapa construimos una tubería (*pipeline*) completa de extremo a extremo:
1. **Lectura (`readStream`):** Spark se conecta al socket TCP `localhost:9999` donde está escuchando **Netcat (`nc -lk 9999`)**.
2. **Transformación:** Cada línea recibida se divide por espacios usando `split()` y se descompone en filas individuales de palabras con `explode()`. Posteriormente se agrupa con `groupBy("palabra").count()`.
3. **Salida (`writeStream`):** Se utiliza `.outputMode("complete")`, lo que indica a Spark que imprima en consola la tabla completa con el conteo histórico acumulado cada vez que llega un nuevo mensaje.

In [3]:
# Caso 1: Hola Spark Streaming (WordCount en tiempo real)
lineasDF = (
    spark.readStream
    .format("socket")
    .option("host", "localhost")
    .option("port", 9999)
    .load()
)

palabrasDF = lineasDF.select(explode(split(lineasDF.value, " ")).alias("palabra"))
cantidadDF = palabrasDF.groupBy("palabra").count()

wordCountQuery = (
    cantidadDF.writeStream
    .format("console")
    .outputMode("complete")
    .start()
)

wordCountQuery.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|streaming|    1|
|    spark|    1|
|     hola|    1|
+---------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|streaming|    1|
|    spark|    2|
|     hola|    2|
|    mundo|    1|
+---------+-----+

-------------------------------------------
Batch: 3
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|   nombre|    1|
|streaming|    1|
|       es|    1|
|    spark|    2|
|     hola|    2|
|       mi|    1|
| valentin|    3|
|    mundo|    1|
+---------+-----+


## 3. Ejercicio 1: Comparación de Modos de Salida (`update` y `append`)

### 3.1. Modo de salida `update`
A diferencia de `complete` (que reimprime toda la tabla en cada lote), el modo **`update`** únicamente emite hacia la consola las filas que son **nuevas** o cuyo conteo **se modificó** durante el micro-batch actual, optimizando el volumen de salida.

In [4]:
# Detener streams previos activos
for q in spark.streams.active:
    q.stop()

# Ejercicio 1: Comparación de modos de salida - Modo "update"
# Solo imprime las palabras nuevas o cuyo conteo cambió en el micro-batch actual
wordCountUpdate = (
    cantidadDF.writeStream
    .format("console")
    .outputMode("update")
    .start()
)

wordCountUpdate.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 4
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|   nombre|    1|
|streaming|    1|
|       es|    1|
|    spark|    3|
|     hola|    2|
|       mi|    1|
|    nuevo|    1|
| valentin|    3|
|    mundo|    1|
|     dato|    1|
+---------+-----+


### 3.2. Modo de salida `append` (Transformaciones sin estado)
El modo **`append`** solo se puede aplicar cuando las filas resultantes **no cambian en el futuro** (transformaciones *stateless* o sin estado, donde no hay agregaciones globales como `groupBy().count()`).

Aquí aplicamos dos transformaciones fila por fila:
- `filter(length("palabra") > 4)`: descarta palabras cortas (como `"sol"`).
- `upper("palabra")`: convierte a mayúsculas cada palabra válida (`APACHE`, `SPARK`, `STREAMING`).

In [6]:
# Detener cualquier stream previo activo en Spark
for q in spark.streams.active:
    q.stop()

# Ejercicio 1 (cont.): Modo de salida "append" (solo para transformaciones sin estado)
from pyspark.sql.functions import upper, length

palabrasLargasDF = (
    palabrasDF
    .filter(length("palabra") > 4)
    .select(upper("palabra").alias("palabra_mayuscula"))
)

appendQuery = (
    palabrasLargasDF.writeStream
    .format("console")
    .outputMode("append")
    .start()
)

appendQuery.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------------+
|palabra_mayuscula|
+-----------------+
+-----------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------------+
|palabra_mayuscula|
+-----------------+
|           APACHE|
|            SPARK|
|        STREAMING|
+-----------------+


## 4. Ejercicio 2: Control de Frecuencia con Triggers (`processingTime`)

Por defecto, Spark dispara un micro-batch tan pronto como termina el anterior. Al configurar `.trigger(processingTime="10 seconds")`, obligamos al motor a esperar **ventanas fijas de 10 segundos** para acumular todos los mensajes enviados durante ese lapso y procesarlos juntos en un único lote.

In [7]:
# Detener streams previos activos
for q in spark.streams.active:
    q.stop()

# Ejercicio 2: Triggers - Intervalo fijo (processingTime = "10 seconds")
wordCountTrigger = (
    cantidadDF.writeStream
    .format("console")
    .outputMode("complete")
    .trigger(processingTime="10 seconds")
    .start()
)

wordCountTrigger.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|   prueba|    1|
|  trigger|    1|
|    batch|    1|
|    spark|    1|
|segundos,|    1|
|     diez|    1|
+---------+-----+


## 5. Cierre Limpio de la Sesión de Spark

Detenemos cualquier consulta activa restante y liberamos los recursos de memoria y hilos de la `SparkSession`.

In [8]:
# Detener cualquier stream activo y cerrar Spark limpiamente
for q in spark.streams.active:
    q.stop()

spark.stop()
print("Sesión de Spark Streaming finalizada correctamente.")

Sesión de Spark Streaming finalizada correctamente.


---

### 📝 Conclusiones
- **Structured Streaming** permite tratar flujos continuos de información usando exactamente las mismas operaciones declarativas de DataFrames (`select`, `filter`, `groupBy`), ocultando la complejidad de la gestión de micro-batches.
- La elección del **`outputMode`** depende directamente de la naturaleza de la consulta: `complete` y `update` son indispensables cuando existen agregaciones con estado (`groupBy().count()`), mientras que `append` está diseñado para transformaciones sin estado donde cada registro emitido es definitivo.
- El uso de **`trigger(processingTime=...)`** permite equilibrar latencia y rendimiento agrupando múltiples eventos dentro de intervalos controlados.